# 🐼 Project 2 - Pandas CSV Reader & Basic Analysis
**Syntecxhub Data Science Internship | Week 1**

---

## 📌 What is Pandas?
**Pandas** is the go-to data manipulation library in Python.  
It provides two key structures:
- **Series** - a 1D labeled array (like a spreadsheet column)
- **DataFrame** - a 2D labeled table (like an Excel sheet or SQL table)

Think of it as **Excel, but programmable and infinitely scalable**.

### This project covers:
1. Reading CSV/Excel into a DataFrame
2. Inspecting data (head, tail, dtypes, info)
3. Computing summary statistics
4. Filtering rows, selecting columns, slicing subsets
5. Saving filtered results to CSV and Excel

---

In [39]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

Pandas version: 2.3.3
NumPy version:  2.4.3


---
## 🗂️ Step 0: Generate Sample Dataset
We create a realistic 200-row student dataset so you can run the notebook without needing an external file.  
*(In a real project, skip this step and load your own CSV.)*

In [40]:
np.random.seed(42)
n = 200

sample_data = {
    "student_id":    [f"S{str(i).zfill(3)}" for i in range(1, n + 1)],
    "name":          np.random.choice(
                         ["Alice", "Bob", "Carlos", "Diana", "Ethan",
                          "Fatima", "Grace", "Hasan", "Isla", "Jamal"], n),
    "age":           np.random.randint(18, 26, n),
    "gender":        np.random.choice(["Male", "Female", "Other"], n,
                                       p=[0.48, 0.48, 0.04]),
    "department":    np.random.choice(
                         ["CS", "Math", "Physics", "Biology", "Business"], n),
    "math_score":    np.random.randint(40, 100, n).astype(float),
    "science_score": np.random.randint(35, 100, n).astype(float),
    "english_score": np.random.randint(45, 100, n).astype(float),
    "attendance_%":  np.round(np.random.uniform(55, 100, n), 1),
    "grade":         np.random.choice(["A", "B", "C", "D", "F"], n,
                                       p=[0.2, 0.3, 0.3, 0.15, 0.05]),
    "enrolled_date": pd.date_range("2022-09-01", periods=n, freq="3D")
                         .strftime("%Y-%m-%d"),
}

df_raw = pd.DataFrame(sample_data)
# Add some missing values (realistic!)
for col in ["math_score", "science_score", "attendance_%"]:
    idx = np.random.choice(df_raw.index, size=10, replace=False)
    df_raw.loc[idx, col] = np.nan

df_raw.to_csv("students.csv", index=False)
print(f"✅ students.csv created — {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

✅ students.csv created — 200 rows × 11 columns


---
## 📥 Section 1: Reading CSV / Excel into a DataFrame

In [41]:
# Read CSV file
df = pd.read_csv("students.csv")

# Other common read methods:
# df = pd.read_excel("file.xlsx", sheet_name="Sheet1")
# df = pd.read_json("file.json")
# df = pd.read_sql("SELECT * FROM table", connection)

print(f"✅ File loaded successfully")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

✅ File loaded successfully
Shape: 200 rows × 11 columns


---
## 🔍 Section 2: Inspecting the DataFrame
Always inspect your data first before doing any analysis - understand what you're working with.

In [42]:
# First 5 rows
print("--- First 5 rows ---")
df.head()

--- First 5 rows ---


,student_id,name,age,gender,department,math_score,science_score,english_score,attendance_%,grade,enrolled_date
0,S001,Grace,23,Male,Physics,87.0,35.0,62.0,56.5,B,2022-09-01
1,S002,Diana,20,Male,Business,56.0,74.0,46.0,86.4,F,2022-09-04
2,S003,Hasan,24,Male,Biology,65.0,98.0,45.0,68.4,A,2022-09-07
3,S004,Ethan,25,Female,CS,75.0,56.0,91.0,96.6,B,2022-09-10
4,S005,Grace,25,Male,Biology,40.0,94.0,49.0,98.7,D,2022-09-13


In [43]:
# Last 3 rows
print("--- Last 3 rows ---")
df.tail(3)

--- Last 3 rows ---


,student_id,name,age,gender,department,math_score,science_score,english_score,attendance_%,grade,enrolled_date
197,S198,Diana,23,Female,CS,51.0,70.0,65.0,58.7,A,2024-04-14
198,S199,Bob,24,Female,CS,74.0,63.0,66.0,85.6,A,2024-04-17
199,S200,Fatima,19,Female,Physics,74.0,94.0,51.0,84.5,B,2024-04-20


In [44]:
# Column names and data types
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Column names: ['student_id', 'name', 'age', 'gender', 'department', 'math_score', 'science_score', 'english_score', 'attendance_%', 'grade', 'enrolled_date']

Data types:
student_id        object
name              object
age                int64
gender            object
department        object
math_score       float64
science_score    float64
english_score    float64
attendance_%     float64
grade             object
enrolled_date     object
dtype: object


In [45]:
# Full overview: shape, dtypes, non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   student_id     200 non-null    object 
 1   name           200 non-null    object 
 2   age            200 non-null    int64  
 3   gender         200 non-null    object 
 4   department     200 non-null    object 
 5   math_score     190 non-null    float64
 6   science_score  190 non-null    float64
 7   english_score  200 non-null    float64
 8   attendance_%   190 non-null    float64
 9   grade          200 non-null    object 
 10  enrolled_date  200 non-null    object 
dtypes: float64(4), int64(1), object(6)
memory usage: 17.3+ KB


In [46]:
# Missing value check
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

Missing values per column:
student_id        0
name              0
age               0
gender            0
department        0
math_score       10
science_score    10
english_score     0
attendance_%     10
grade             0
enrolled_date     0
dtype: int64

Total missing cells: 30


---
## 📊 Section 3: Summary Statistics
`.describe()` gives you a quick statistical overview of all numeric columns at once.

In [47]:
# Auto-summary for all numeric columns
df.describe().round(2)

,age,math_score,science_score,english_score,attendance_%
count,200.00,190.00,190.00,200.00,190.00
mean,21.64,68.48,67.48,72.62,78.92
std,2.43,17.62,18.29,15.42,13.67
min,18.00,40.00,35.00,45.00,55.30
25%,20.00,53.00,53.00,61.00,66.70
50%,21.00,69.00,67.50,71.00,81.70
75%,24.00,84.00,83.00,86.00,90.88
max,25.00,99.00,99.00,99.00,99.90


In [48]:
# Manual stats for math_score
col = df["math_score"].dropna()
print("--- Math Score Statistics ---")
print(f"  Mean:    {col.mean():.2f}")
print(f"  Median:  {col.median():.2f}")
print(f"  Mode:    {col.mode().values[0]}")
print(f"  Min:     {col.min():.1f}")
print(f"  Max:     {col.max():.1f}")
print(f"  Count:   {col.count()}  (missing: {df['math_score'].isna().sum()})")
print(f"  Std Dev: {col.std():.2f}")

--- Math Score Statistics ---
  Mean:    68.48
  Median:  69.00
  Mode:    65.0
  Min:     40.0
  Max:     99.0
  Count:   190  (missing: 10)
  Std Dev: 17.62


In [49]:
# Categorical distributions
print("Grade distribution:")
print(df["grade"].value_counts())
print("\nDepartment distribution:")
print(df["department"].value_counts())

Grade distribution:
grade
B    61
C    55
A    45
D    28
F    11
Name: count, dtype: int64

Department distribution:
department
Business    47
CS          44
Math        40
Physics     37
Biology     32
Name: count, dtype: int64


In [50]:
# Grouped mean scores by department
score_cols = ["math_score", "science_score", "english_score"]
print("Average scores by department:")
df.groupby("department")[score_cols].mean().round(2)

Average scores by department:


,math_score,science_score,english_score
department,,,
Biology,65.13,70.31,71.88
Business,69.86,69.72,68.53
CS,70.37,64.38,76.80
Math,69.29,66.89,71.82
Physics,66.57,66.06,74.35


---
## 🎯 Section 4: Filtering, Column Selection & Slicing

| Method | Use case |
|---|---|
| `df["col"]` | Select single column |
| `df[["col1","col2"]]` | Select multiple columns |
| `df[condition]` | Filter rows by condition |
| `df.iloc[rows, cols]` | Slice by integer position |
| `df.loc[condition, cols]` | Slice by label/condition |

In [51]:
# Select multiple columns
subset = df[["student_id", "name", "math_score", "grade"]]
print(f"Subset shape: {subset.shape}")
subset.head()

Subset shape: (200, 4)


,student_id,name,math_score,grade
0,S001,Grace,87.0,B
1,S002,Diana,56.0,F
2,S003,Hasan,65.0,A
3,S004,Ethan,75.0,B
4,S005,Grace,40.0,D


In [52]:
# Filter: math score >= 85
high_achievers = df[df["math_score"] >= 85]
print(f"Students with math_score ≥ 85: {len(high_achievers)} rows")
high_achievers[["student_id", "name", "math_score", "grade"]].head()

Students with math_score ≥ 85: 45 rows


,student_id,name,math_score,grade
0,S001,Grace,87.0,B
6,S007,Carlos,88.0,C
8,S009,Hasan,91.0,B
10,S011,Diana,86.0,B
11,S012,Hasan,95.0,D


In [53]:
# Filter: multiple conditions (& = AND, | = OR)
cs_grade_a = df[(df["department"] == "CS") & (df["grade"] == "A")]
print(f"CS students with Grade A: {len(cs_grade_a)} rows")
cs_grade_a[["student_id", "name", "department", "grade"]].head()

CS students with Grade A: 11 rows


,student_id,name,department,grade
27,S028,Carlos,CS,A
64,S065,Bob,CS,A
72,S073,Bob,CS,A
78,S079,Alice,CS,A
81,S082,Isla,CS,A


In [54]:
# isin() — filter by a list of values
top_depts = df[df["department"].isin(["CS", "Math"])]
print(f"CS & Math students: {len(top_depts)} rows")
top_depts["department"].value_counts()

CS & Math students: 84 rows


department
CS      44
Math    40
Name: count, dtype: int64

In [55]:
# .iloc — slice by position
print("--- .iloc: rows 10-14, first 4 columns ---")
print(df.iloc[10:15, :4])

# .loc — slice by label/condition
print("\n--- .loc: students with attendance < 65% ---")
low_att = df.loc[df["attendance_%"] < 65,
                 ["student_id", "name", "attendance_%", "grade"]]
print(f"Total low-attendance students: {len(low_att)}")
low_att.head()

--- .iloc: rows 10-14, first 4 columns ---
   student_id    name  age  gender
10       S011   Diana   25  Female
11       S012   Hasan   22    Male
12       S013   Hasan   20    Male
13       S014  Carlos   21  Female
14       S015  Fatima   21    Male

--- .loc: students with attendance < 65% ---
Total low-attendance students: 39


,student_id,name,attendance_%,grade
0,S001,Grace,56.5,B
11,S012,Hasan,56.7,D
14,S015,Fatima,60.4,A
19,S020,Bob,57.9,F
28,S029,Grace,56.1,A


---
## 💾 Section 5: Saving Filtered Results

In [56]:
# Save CS students to CSV
cs_students = df[df["department"] == "CS"].copy()
cs_students.to_csv("cs_students.csv", index=False)
print(f"✅ Saved CS students ({len(cs_students)} rows) → cs_students.csv")

# Save summary stats to CSV
summary = df[score_cols].describe().round(2)
summary.to_csv("summary_stats.csv")
print(f"✅ Saved summary stats → summary_stats.csv")

✅ Saved CS students (44 rows) → cs_students.csv
✅ Saved summary stats → summary_stats.csv


In [57]:
# Save multi-sheet Excel file
try:
    with pd.ExcelWriter("analysis_results.xlsx", engine="openpyxl") as writer:
        df.to_excel(writer,              sheet_name="All Students",  index=False)
        cs_students.to_excel(writer,     sheet_name="CS Department", index=False)
        high_achievers.to_excel(writer,  sheet_name="High Achievers",index=False)
        summary.to_excel(writer,         sheet_name="Summary Stats")
    print("✅ Saved multi-sheet Excel → analysis_results.xlsx")
except ImportError:
    print("openpyxl not installed — install with: pip install openpyxl")

✅ Saved multi-sheet Excel → analysis_results.xlsx


---
## ✅ Summary

| Task | Method Used |
|---|---|
| Read CSV | `pd.read_csv()` |
| Inspect data | `.head()`, `.tail()`, `.info()`, `.dtypes` |
| Summary stats | `.describe()`, `.mean()`, `.median()`, `.groupby()` |
| Filter rows | `df[condition]`, `&`, `isin()` |
| Slice subsets | `.iloc[]`, `.loc[]` |
| Save results | `.to_csv()`, `.to_excel()` |

---
*Syntecxhub Data Science Internship - Week 1, Project 2*